# kpi_metrics — Usage Examples

This notebook walks through the core components of `kpi_metrics` using a synthetic dataset of 5 companies with 3 years of daily transaction history.

## 0. Generate Test Dataset

5 companies × 3 years (2021-01-01 → 2023-12-31) of daily observations.
Metrics: `SALES` (daily revenue) and `COUNT` (daily transaction count).

In [1]:
import polars as pl
import numpy as np
from datetime import date, timedelta

rng = np.random.default_rng(42)

companies = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]
start = date(2021, 1, 1)
n_days = 3 * 365  # 1095 days
dates = [start + timedelta(days=i) for i in range(n_days)]

# Each company gets its own baseline + trend + noise
base_sales  = {"AAPL": 500, "MSFT": 400, "GOOGL": 350, "AMZN": 600, "META": 300}
base_count  = {"AAPL": 100, "MSFT":  80, "GOOGL":  70, "AMZN": 120, "META":  60}
growth_rate = {"AAPL": 0.10, "MSFT": 0.08, "GOOGL": 0.12, "AMZN": 0.15, "META": 0.05}

rows = []
for co in companies:
    for i, d in enumerate(dates):
        t = i / n_days
        trend = 1 + growth_rate[co] * t * 3  # linear annual growth over 3 years
        sales = base_sales[co] * trend * (1 + 0.15 * rng.standard_normal())
        count = int(base_count[co] * trend * (1 + 0.10 * rng.standard_normal()))
        rows.append({"specid": co, "DATADATE": d, "SALES": round(float(sales), 2), "COUNT": float(max(count, 1))})

raw_df = pl.DataFrame(rows).sort(["specid", "DATADATE"])

print(f"Shape: {raw_df.shape}")
print(f"Companies: {raw_df['specid'].unique().to_list()}")
print(f"Date range: {raw_df['DATADATE'].min()} → {raw_df['DATADATE'].max()}")
raw_df.head(10)

Shape: (5475, 4)
Companies: ['AAPL', 'MSFT', 'META', 'AMZN', 'GOOGL']
Date range: 2021-01-01 → 2023-12-31


specid,DATADATE,SALES,COUNT
str,date,f64,f64
"""AAPL""",2021-01-01,522.85,89.0
"""AAPL""",2021-01-02,556.44,109.0
"""AAPL""",2021-01-03,353.87,87.0
"""AAPL""",2021-01-04,510.01,96.0
"""AAPL""",2021-01-05,499.29,91.0
"""AAPL""",2021-01-06,566.73,107.0
"""AAPL""",2021-01-07,505.78,111.0
"""AAPL""",2021-01-08,536.09,91.0
"""AAPL""",2021-01-09,528.81,90.0


## 1. Params

In [2]:
from kpi_metrics import Param

params = Param(
    keys=["specid"],
    metrics=["SALES", "COUNT"],
)
print(params)

Param(keys=['specid'], metrics=['SALES', 'COUNT'], date_col='DATADATE')


## 2. Base Features

### 2a. Rolling Sum

In [3]:
from kpi_metrics import RollingSumBaseFeature

rolling = RollingSumBaseFeature(params, windows=[("weekly", 7), ("monthly", 28), ("quarterly", 91)])
rolling_df = rolling.compute(raw_df)

print("Columns:", rolling_df.columns)
rolling_df.filter(pl.col("specid") == "AAPL").tail(5)

Columns: ['specid', 'DATADATE', 'SALES__ROLLING_SUM__WEEKLY', 'SALES__ROLLING_SUM__MONTHLY', 'SALES__ROLLING_SUM__QUARTERLY', 'COUNT__ROLLING_SUM__WEEKLY', 'COUNT__ROLLING_SUM__MONTHLY', 'COUNT__ROLLING_SUM__QUARTERLY']


specid,DATADATE,SALES__ROLLING_SUM__WEEKLY,SALES__ROLLING_SUM__MONTHLY,SALES__ROLLING_SUM__QUARTERLY,COUNT__ROLLING_SUM__WEEKLY,COUNT__ROLLING_SUM__MONTHLY,COUNT__ROLLING_SUM__QUARTERLY
str,date,f64,f64,f64,f64,f64,f64
"""AAPL""",2023-12-27,4703.33,18991.27,59491.85,871.0,3621.0,11626.0
"""AAPL""",2023-12-28,4594.67,18599.98,59366.06,866.0,3615.0,11624.0
"""AAPL""",2023-12-29,4508.82,18393.32,59167.46,881.0,3603.0,11624.0
"""AAPL""",2023-12-30,4527.07,18375.79,59302.34,891.0,3567.0,11606.0
"""AAPL""",2023-12-31,4691.44,18612.29,59519.84,898.0,3571.0,11630.0


### 2b. YoY Growth

In [4]:
from kpi_metrics import YOYGrowthBaseFeature

yoy = YOYGrowthBaseFeature(params, windows=[("monthly", 28), ("quarterly", 91)])
yoy_df = yoy.compute(raw_df)

print("Columns:", yoy_df.columns)
# First year will be null (no prior-year data yet)
yoy_df.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY', 'SALES__YOY_GROWTH__QUARTERLY', 'COUNT__YOY_GROWTH__MONTHLY', 'COUNT__YOY_GROWTH__QUARTERLY']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY,SALES__YOY_GROWTH__QUARTERLY,COUNT__YOY_GROWTH__MONTHLY,COUNT__YOY_GROWTH__QUARTERLY
str,date,f64,f64,f64,f64
"""AAPL""",2021-12-31,27.625954,92.079277,32.876404,107.898876
"""AAPL""",2022-01-01,12.841859,44.084185,14.212121,47.929293
"""AAPL""",2022-01-02,9.322127,32.970813,9.547368,32.97193
"""AAPL""",2022-01-03,6.610384,24.060659,6.900262,24.364829
"""AAPL""",2022-01-04,5.055104,18.972004,5.381356,19.504237


### 2c. Period-over-Period Growth

In [5]:
from kpi_metrics import POPGrowthBaseFeature

pop = POPGrowthBaseFeature(params, windows=[("monthly", 28), ("quarterly", 91)])
pop_df = pop.compute(raw_df)

print("Columns:", pop_df.columns)
pop_df.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Columns: ['specid', 'DATADATE', 'SALES__POP_GROWTH__MONTHLY', 'COUNT__POP_GROWTH__MONTHLY', 'SALES__POP_GROWTH__QUARTERLY', 'COUNT__POP_GROWTH__QUARTERLY']


specid,DATADATE,SALES__POP_GROWTH__MONTHLY,COUNT__POP_GROWTH__MONTHLY,SALES__POP_GROWTH__QUARTERLY,COUNT__POP_GROWTH__QUARTERLY
str,date,f64,f64,f64,f64
"""AAPL""",2021-04-02,0.015366,0.01581,86.841809,101.41573
"""AAPL""",2021-04-03,0.019688,0.030336,41.543913,45.080808
"""AAPL""",2021-04-04,0.025775,0.038559,31.159829,31.021053
"""AAPL""",2021-04-05,0.008942,0.046977,22.666375,23.013123
"""AAPL""",2021-04-06,0.038134,0.052555,17.888395,18.402542


## 3. Benchmarks

### 3a. Zero Benchmark

In [6]:
from kpi_metrics import ZeroBenchmark

zero_bench = ZeroBenchmark(params)
zero_df = zero_bench.compute(yoy_df, raw_df)
zero_df.head(5)

specid,DATADATE,ZERO_BENCHMARK
str,date,f64
"""AAPL""",2021-01-01,0.0
"""AAPL""",2021-01-02,0.0
"""AAPL""",2021-01-03,0.0
"""AAPL""",2021-01-04,0.0
"""AAPL""",2021-01-05,0.0


### 3b. Lagged YoY Growth Benchmark

In [7]:
from kpi_metrics import LaggedYOYGrowthBenchmark

lagged_bench = LaggedYOYGrowthBenchmark(params, lags=[28, 91])
lagged_df = lagged_bench.compute(yoy_df, raw_df)

print("Columns:", lagged_df.columns)
lagged_df.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK', 'SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK', 'COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK', 'COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK', 'SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK', 'SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK', 'COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK', 'COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK,SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK,COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK,COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK,SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK,SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK,COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK,COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK
str,date,f64,f64,f64,f64,f64,f64,f64,f64
"""AAPL""",2022-04-01,0.140506,0.572411,0.104563,0.568606,27.625954,92.079277,32.876404,107.898876
"""AAPL""",2022-04-02,0.145802,0.549444,0.116649,0.546761,12.841859,44.084185,14.212121,47.929293
"""AAPL""",2022-04-03,0.135423,0.516667,0.126228,0.527151,9.322127,32.970813,9.547368,32.97193
"""AAPL""",2022-04-04,0.115388,0.48859,0.117261,0.50263,6.610384,24.060659,6.900262,24.364829
"""AAPL""",2022-04-05,0.128604,0.469544,0.125182,0.481767,5.055104,18.972004,5.381356,19.504237


### 3c. Cohort Rolling Sum Benchmark

Compare each company against the mean of its cohort. Here we group all 5 into a single `"ALL"` cohort for illustration.

In [8]:
from kpi_metrics import CohortRollingSumBenchmark

# Attach a cohort label to raw_df
cohort_raw = raw_df.with_columns(pl.lit("ALL").alias("cohort"))

rolling_for_cohort = RollingSumBaseFeature(params, windows=[("monthly", 28)]).compute(cohort_raw)
rolling_for_cohort = rolling_for_cohort.join(
    cohort_raw.select(["specid", "DATADATE", "cohort"]), on=["specid", "DATADATE"]
)

cohort_bench = CohortRollingSumBenchmark(params, cohort_cols=["cohort"], agg="mean")
cohort_df = cohort_bench.compute(rolling_for_cohort, cohort_raw)

print("Columns:", cohort_df.columns)
cohort_df.filter(pl.col("specid") == "AAPL").head(5)

Columns: ['specid', 'DATADATE', 'SALES__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK', 'COUNT__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK']


specid,DATADATE,SALES__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK,COUNT__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK
str,date,f64,f64
"""AAPL""",2021-01-01,463.772,85.6
"""AAPL""",2021-01-02,934.762,168.6
"""AAPL""",2021-01-03,1349.804,248.4
"""AAPL""",2021-01-04,1791.428,330.2
"""AAPL""",2021-01-05,2168.516,416.4


## 4. Comparisons

### 4a. Difference & Sign

In [9]:
from kpi_metrics import DifferenceComparison, SignComparison

diff = DifferenceComparison().compute(yoy_df, zero_df)
sign = SignComparison().compute(yoy_df, zero_df)

print("Difference columns:", diff.columns)
diff.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Difference columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE
str,date,f64,f64,f64,f64
"""AAPL""",2021-12-31,27.625954,92.079277,32.876404,107.898876
"""AAPL""",2022-01-01,12.841859,44.084185,14.212121,47.929293
"""AAPL""",2022-01-02,9.322127,32.970813,9.547368,32.97193
"""AAPL""",2022-01-03,6.610384,24.060659,6.900262,24.364829
"""AAPL""",2022-01-04,5.055104,18.972004,5.381356,19.504237


In [10]:
print("Sign columns:", sign.columns)
sign.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Sign columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN
str,date,f64,f64,f64,f64
"""AAPL""",2021-12-31,1.0,1.0,1.0,1.0
"""AAPL""",2022-01-01,1.0,1.0,1.0,1.0
"""AAPL""",2022-01-02,1.0,1.0,1.0,1.0
"""AAPL""",2022-01-03,1.0,1.0,1.0,1.0
"""AAPL""",2022-01-04,1.0,1.0,1.0,1.0


### 4b. Rolling Z-Score

In [11]:
from kpi_metrics import ZScoreComparison

zscore = ZScoreComparison(params, lookback_window=91).compute(yoy_df, zero_df)

print("ZScore columns:", zscore.columns)
zscore.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

ZScore columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE', 'COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE
str,date,f64,f64,f64,f64
"""AAPL""",2022-03-31,-0.293939,-0.374284,-0.266429,-0.352397
"""AAPL""",2022-04-01,-0.364148,-0.488089,-0.340182,-0.482847
"""AAPL""",2022-04-02,-0.3967,-0.542089,-0.376993,-0.548553
"""AAPL""",2022-04-03,-0.41828,-0.600379,-0.398317,-0.6017
"""AAPL""",2022-04-04,-0.413884,-0.644827,-0.413704,-0.644186


### 4c. Cross-Sectional Rank

In [12]:
from kpi_metrics import RankComparison

rank = RankComparison(params, lookback_window=91).compute(yoy_df, zero_df)

# Rank values are in [-1, 1]
rank_col = [c for c in rank.columns if "__RANK" in c][0]
print(f"Column: {rank_col}")
print(f"Range: [{rank[rank_col].drop_nulls().min():.3f}, {rank[rank_col].drop_nulls().max():.3f}]")
rank.drop_nulls().head(5)

Column: SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__RANK
Range: [-0.978, 1.000]


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__RANK,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__RANK,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__RANK,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__RANK
str,date,f64,f64,f64,f64
"""AAPL""",2022-03-31,-0.978022,-0.978022,-0.758242,-0.978022
"""AAPL""",2022-04-01,-0.956044,-0.956044,-0.89011,-0.978022
"""AAPL""",2022-04-02,-0.978022,-0.934066,-0.978022,-0.978022
"""AAPL""",2022-04-03,-0.868132,-0.978022,-0.956044,-0.934066
"""AAPL""",2022-04-04,-0.736264,-0.89011,-0.934066,-0.978022


### 4d. OLS Residual

In [13]:
from kpi_metrics import ResidualComparison

# Use lagged YoY as benchmark to regress out the trend
residual = ResidualComparison(params, lookback_window=91).compute(yoy_df, lagged_df)

print("Residual columns:", residual.columns)
residual.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Residual columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__RESIDUAL', 'SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__RESIDUAL', 'SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__RESIDUAL', 'SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__M

specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__RESIDUAL
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AAPL""",2022-06-30,-0.063135,-0.039325,0.022126,-0.039389,-0.041342,-0.040748,-0.041133,-0.040755,-0.020852,-0.012155,0.000383,-0.013681,-0.015746,-0.01516,-0.015801,-0.015293,0.013039,-0.004889,0.000888,-0.000653,0.004436,0.003069,0.004778,0.003531,0.012852,0.010422,0.002217,0.010945,0.011576,0.011371,0.011594,0.011418
"""AAPL""",2022-07-01,-0.054376,-0.035227,0.011693,-0.035819,-0.038409,-0.036459,-0.038253,-0.036517,-0.017324,-0.009598,-0.00055,-0.010972,-0.012861,-0.011595,-0.012876,-0.011624,0.015801,-0.000879,0.005681,0.002664,0.006368,0.003658,0.006612,0.003792,0.011628,0.009463,0.003223,0.009981,0.010542,0.010066,0.010555,0.010081
"""AAPL""",2022-07-02,-0.052932,-0.031515,0.014338,-0.032878,-0.035393,-0.032026,-0.035253,-0.031888,-0.017498,-0.00912,0.000125,-0.01076,-0.012485,-0.010711,-0.012449,-0.0106,0.023514,0.005645,0.011239,0.009543,0.012413,0.008985,0.012508,0.008785,0.016486,0.014244,0.007651,0.014896,0.015401,0.01471,0.015403,0.014675
"""AAPL""",2022-07-03,-0.034832,-0.007099,0.033531,-0.009094,-

## 5. Transformations

### 5a. Winsorize

In [14]:
from kpi_metrics import WinsorizeTransformation

winsorized = WinsorizeTransformation(lower_q=0.01, upper_q=0.99).compute(diff)

feat_col = [c for c in diff.columns if "__VS__" in c][0]
print(f"Column: {feat_col}")
print(f"Before winsorize: min={diff[feat_col].drop_nulls().min():.4f}, max={diff[feat_col].drop_nulls().max():.4f}")
print(f"After  winsorize: min={winsorized[feat_col].drop_nulls().min():.4f}, max={winsorized[feat_col].drop_nulls().max():.4f}")

Column: SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
Before winsorize: min=-0.0857, max=34.9344
After  winsorize: min=-0.0106, max=2.7568


### 5b. Ramp Up

Useful for down-weighting new entities in their early days. Here we simulate an `age_days` column.

In [15]:
from kpi_metrics import RampUpTransformation

# Attach age (days since first observation) to the comparison output
age_lookup = (
    raw_df.group_by("specid").agg(pl.col("DATADATE").min().alias("first_date"))
)
diff_with_age = (
    diff
    .join(age_lookup, on="specid")
    .with_columns(
        ((pl.col("DATADATE") - pl.col("first_date")).dt.total_days()).alias("age_days")
    )
    .drop("first_date")
)

ramped = RampUpTransformation(age_col="age_days", min_age=0, max_age=365).compute(diff_with_age)

feat_col = [c for c in diff.columns if "__VS__" in c][0]
# Features in the first year are scaled down from 0→1
ramped.select(["specid", "DATADATE", "age_days", feat_col]).filter(
    pl.col("specid") == "AAPL"
).drop_nulls().head(8)

specid,DATADATE,age_days,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
str,date,i64,f64
"""AAPL""",2021-12-31,364,27.550266
"""AAPL""",2022-01-01,365,12.841859
"""AAPL""",2022-01-02,366,9.322127
"""AAPL""",2022-01-03,367,6.610384
"""AAPL""",2022-01-04,368,5.055104
"""AAPL""",2022-01-05,369,3.837438
"""AAPL""",2022-01-06,370,3.176158
"""AAPL""",2022-01-07,371,2.587626


## 6. Full Pipeline with KpiMetricsBuilder

Combine everything into a single, reusable builder.

In [16]:
from kpi_metrics import (
    KpiMetricsBuilder, Param,
    YOYGrowthBaseFeature,
    ZeroBenchmark, LaggedYOYGrowthBenchmark,
    DifferenceComparison, ZScoreComparison,
    WinsorizeTransformation,
)

params = Param(keys=["specid"], metrics=["SALES", "COUNT"])

builder = KpiMetricsBuilder(
    params=params,
    base_feature=YOYGrowthBaseFeature(params, windows=[("monthly", 28), ("quarterly", 91)]),
    benchmarks=[
        ZeroBenchmark(params),
        LaggedYOYGrowthBenchmark(params, lags=[28, 91]),
    ],
    comparisons=[
        DifferenceComparison(),
        ZScoreComparison(params, lookback_window=91),
    ],
    transformations=[
        WinsorizeTransformation(lower_q=0.01, upper_q=0.99),
    ],
)

result = builder.compute(raw_df)

print(f"Output shape: {result.shape}")
print(f"\nOutput columns ({len(result.columns)} total):")
for col in result.columns:
    print(" ", col)

Output shape: (5475, 74)

Output columns (74 total):
  specid
  DATADATE
  SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
  SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE
  COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
  COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE
  SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE
  COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE
  COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE
  SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFF

In [17]:
# Preview results for AAPL (drop rows with nulls from warmup period)
result.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GRO

In [18]:
# Summary statistics across all companies
feat_cols = [c for c in result.columns if "__VS__" in c]
result.select(feat_cols).drop_nulls().describe()

statistic,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG28_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG28_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG91_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG91_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__Q

## 7. Preset Builders

For common use cases, presets provide ready-to-use builders with sensible defaults.

In [19]:
from kpi_metrics.presets import sales_yoy_vs_zero, sales_rolling_vs_zero

params2 = Param(keys=["specid"], metrics=["SALES", "COUNT"])

# YoY growth vs. zero (difference + sign)
yoy_result = sales_yoy_vs_zero(params2).compute(raw_df)
print("sales_yoy_vs_zero columns:", yoy_result.columns)

# Rolling sum vs. zero (difference + sign)
roll_result = sales_rolling_vs_zero(params2).compute(raw_df)
print("\nsales_rolling_vs_zero columns:", roll_result.columns)

sales_yoy_vs_zero columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__SIGN', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN']

sales_rolling_vs_zero columns: ['specid', 'DATADATE', 'SALES__ROLLING_SUM__DAILY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__ROLLING_SUM__WEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__ROLLING_SUM__QUADWEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__ROLLING_SUM__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__ROLLING_SUM__DAILY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__ROLLING_SUM__WEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__ROLLING_SUM__QUADW